In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import LabelEncoder ,StandardScaler # Import LabelEncoder
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score ,classification_report
from sklearn.preprocessing import StandardScaler ,LabelEncoder
from sklearn.metrics import confusion_matrix


In [ ]:
data=pd.read_csv('/content/Breast_Cancer.csv')

# explore data

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
data.describe()

# Analysis

In [ ]:
sns.countplot(x='Status',data=data, palette=(['g','r']))

In [ ]:
from imblearn.over_sampling import RandomOverSampler

In [ ]:
data.value_counts('Status')

# transform data

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
x = data.drop(columns=['Status'])
y = data['Status']
if y.isna().any():
    raise ValueError('Status contains missing labels')

In [ ]:
x_train_raw, x_test_raw, y_train_raw, y_test_raw = train_test_split(x, y, test_size=0.2, random_state=22, stratify=y)

In [ ]:
cat = x_train_raw.select_dtypes(include=['object', 'category', 'string', 'bool']).columns.tolist()
num = [c for c in x_train_raw.columns if c not in cat]

In [ ]:
preprocess = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num),
    ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat),
])

In [ ]:
x_train = preprocess.fit_transform(x_train_raw)
x_test = preprocess.transform(x_test_raw)

In [ ]:
la = LabelEncoder()
y_train = la.fit_transform(y_train_raw)
y_test = la.transform(y_test_raw)

In [ ]:
print('Training and holdout shapes:', x_train.shape, x_test.shape)

# Model

In [ ]:
print('Training labels:', pd.Series(y_train).value_counts().to_dict())

In [ ]:
rs = RandomOverSampler(random_state=41)
x_res, y_res = rs.fit_resample(x_train, y_train)

In [ ]:
assert len(x_test) == len(y_test_raw)
print('Resampled training shape:', x_res.shape)

In [ ]:
model1 = LogisticRegression(max_iter=1000)
model2 = RandomForestClassifier(random_state=22)
model3 = GradientBoostingClassifier(random_state=22)
model4 = DecisionTreeClassifier(random_state=22)
model5 = KNeighborsClassifier()
model6 = GaussianNB()
model7 = SVC()
model8 = XGBClassifier(random_state=22)

In [ ]:
def pred(model):
    model.fit(x_res, y_res)
    pre = model.predict(x_test)
    print(classification_report(y_test, pre, labels=np.arange(len(la.classes_)), target_names=[str(c) for c in la.classes_], zero_division=0))

In [ ]:
pred (model1)

In [ ]:
pred (model2)

In [ ]:
pred (model3)

In [ ]:
pred (model4)

In [ ]:
pred (model5)

In [ ]:
pred (model6)

In [ ]:
pred (model7)